# Tabular Q-Learning

Q-Learning is a **model-free** RL algorithm that learns the optimal action-value function $Q^*(s,a)$ directly from experience. This notebook:
1. Implements tabular Q-learning on a GridWorld
2. Demonstrates exploration vs exploitation ($\varepsilon$-greedy)
3. Visualises learning dynamics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

## 1. Q-Learning Algorithm

The Q-learning update rule (off-policy, TD(0)):
$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

- $\alpha$: learning rate
- $\gamma$: discount factor
- The agent does **not** need the transition model $P$.

In [ ]:
class GridWorld:
    ACTIONS = ['up', 'down', 'left', 'right']
    DELTAS = {'up': (-1,0), 'down': (1,0), 'left': (0,-1), 'right': (0,1)}
    
    def __init__(self, size=5, goal=(0,4), trap=(1,4), obstacles=None):
        self.size = size
        self.goal = goal
        self.trap = trap
        self.obstacles = set(obstacles or [(2,2)])
        self.state = None
    
    def reset(self):
        self.state = (self.size-1, 0)
        return self.state
    
    def step(self, action):
        dr, dc = self.DELTAS[action]
        nr, nc = self.state[0]+dr, self.state[1]+dc
        nxt = (nr, nc)
        if 0<=nr<self.size and 0<=nc<self.size and nxt not in self.obstacles:
            self.state = nxt
        if self.state == self.goal:
            return self.state, 10.0, True
        elif self.state == self.trap:
            return self.state, -10.0, True
        return self.state, -0.1, False

env = GridWorld()
print(f"Grid: {env.size}x{env.size}, Goal: {env.goal}, Trap: {env.trap}")

In [ ]:
def q_learning(env, episodes=2000, alpha=0.1, gamma=0.99, epsilon=0.1):
    Q = defaultdict(lambda: np.zeros(len(env.ACTIONS)))
    rewards_per_episode = []
    
    for ep in range(episodes):
        state = env.reset()
        total_reward = 0
        
        for _ in range(200):  # max steps
            # Epsilon-greedy action selection
            if np.random.rand() < epsilon:
                action_idx = np.random.randint(len(env.ACTIONS))
            else:
                action_idx = np.argmax(Q[state])
            action = env.ACTIONS[action_idx]
            
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # Q-learning update
            best_next = np.max(Q[next_state])
            Q[state][action_idx] += alpha * (reward + gamma * best_next - Q[state][action_idx])
            
            state = next_state
            if done:
                break
        
        rewards_per_episode.append(total_reward)
    
    return dict(Q), rewards_per_episode

Q, rewards = q_learning(env)
print(f"Training complete. Final 100-episode avg reward: {np.mean(rewards[-100:]):.2f}")

In [ ]:
# Plot learning curve
window = 50
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
plt.plot(smoothed)
plt.xlabel('Episode')
plt.ylabel(f'Reward (moving avg, window={window})')
plt.title('Q-Learning: Reward over Episodes')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualise learned policy
arrow_map = {'up': '\u2191', 'down': '\u2193', 'left': '\u2190', 'right': '\u2192'}

V_grid = np.full((env.size, env.size), np.nan)
fig, ax = plt.subplots(figsize=(6, 6))

for r in range(env.size):
    for c in range(env.size):
        s = (r, c)
        if s in env.obstacles:
            ax.text(c, r, '\u2588', ha='center', va='center', fontsize=20, color='black')
        elif s == env.goal:
            ax.text(c, r, 'G', ha='center', va='center', fontsize=16, color='green', weight='bold')
            V_grid[r, c] = np.max(Q.get(s, [0]))
        elif s == env.trap:
            ax.text(c, r, 'T', ha='center', va='center', fontsize=16, color='red', weight='bold')
            V_grid[r, c] = np.max(Q.get(s, [0]))
        elif s in Q:
            best_a = env.ACTIONS[np.argmax(Q[s])]
            V_grid[r, c] = np.max(Q[s])
            ax.text(c, r, arrow_map[best_a], ha='center', va='center', fontsize=18)

ax.imshow(V_grid, cmap='RdYlGn', interpolation='nearest', alpha=0.4)
ax.set_title('Learned Q-Policy')
ax.set_xticks(range(env.size))
ax.set_yticks(range(env.size))
plt.tight_layout()
plt.show()

In [ ]:
# Effect of epsilon on learning
fig, ax = plt.subplots()
for eps in [0.01, 0.1, 0.3, 0.5]:
    _, r = q_learning(env, episodes=1000, epsilon=eps)
    smoothed = np.convolve(r, np.ones(50)/50, mode='valid')
    ax.plot(smoothed, label=f'eps={eps}')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('Effect of Epsilon on Q-Learning')
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

- Q-learning is **off-policy** and **model-free**: it learns $Q^*$ without knowing $P$.
- **Epsilon-greedy** balances exploration and exploitation.
- Tabular Q-learning works for small state spaces; for large/continuous spaces, we need **function approximation** (next: DQN).

**Next:** Deep Q-Networks with PyTorch.